# H2 ground state calculation

In [1]:
import numpy as np
import dmrg
import veloxchem as vlx
np.set_printoptions(suppress=True)
import time as tm
E_FCI = -1.1372744056952255

In [2]:
mol_xyz = """2

H 0.000 0.000 0.000
H 0.000 0.000 0.741
"""

In [ ]:
# Set up the VeloxChem calculation of the HF ground state
molecule = vlx.Molecule.read_xyz_string(mol_xyz)
basis_str = 'sto-3g'
basis = vlx.MolecularBasis.read(molecule, basis_str)
print(basis.info_str(basis_str))

                  Molecular Basis (sto-3g)                  

Basis: STO-3G                                               

  Atom Contracted GTOs           Primitive GTOs                

  H     (1S)                      (3S)                          

Contracted Basis Functions : 2                              
Primitive Basis Functions  : 6                              




In [4]:
scf_drv = vlx.ScfRestrictedDriver()
scf_drv.ostream.mute()
scf_res = scf_drv.compute(molecule,basis)

In [ ]:
# Get the integrals from Veloxchem
int_drv = dmrg.IntegralsDriver()
t_ij, v_ijkl = int_drv.get_transformed_integrals(molecule, basis, scf_res)

In [ ]:
# The number of MOs defines the number of sites
C = scf_res['C_alpha']
m_bonddim = 4
nr_sites = C.shape[1]

In [ ]:
# Set up the DMRG calculation, both a fixed with 0.5 + 0.5i entries and a random MPS may be used for an initial guess
settings = dmrg.Settings(nr_sites=nr_sites, max_bond_dim=m_bonddim)

mpo_drv = dmrg.MpoDriver(settings)
mps_drv = dmrg.MpsDriver(settings)

elec_mpo = mpo_drv.electronic_mpo(t_ij, v_ijkl)

#mps_drv._initialize_fixed_mps()
mps_drv._initialize_random_mps()
mps_drv.canonical_form(0)
mps_drv.normalize()
mps = mps_drv.mps

In [8]:
sweep_drv = dmrg.SweepDriver(settings, mps_drv=mps_drv, mpo_drv=mpo_drv)

In [9]:
E0, _mps = sweep_drv.compute(elec_mpo)

Sweep: 1
Energy after right sweep: -1.851414 a.u.
Discarded weight: max = 0.000e+00, mean = 0.000e+00 (worst bond: 0)

Energy after left sweep : -1.851414 a.u.
Discarded weight: max = 0.000e+00, mean = 0.000e+00 (worst bond: 0)

Sweep: 2
Energy after right sweep: -1.851414 a.u.
Discarded weight: max = 0.000e+00, mean = 0.000e+00 (worst bond: 0)

Energy after left sweep : -1.851414 a.u.
Discarded weight: max = 0.000e+00, mean = 0.000e+00 (worst bond: 0)


** Converged after 2 sweeps! **
Ground-state energy = -1.851414 a.u.



In [10]:
print(f'DMRG: {(E0 + molecule.nuclear_repulsion_energy()):.6f}')
print(f'HF  : {scf_res['scf_energy']:.6f}')
print(f'FCI : {E_FCI:.6f}')
print(f'\nFCI - HF: {abs(E_FCI-scf_res['scf_energy']):.2e}')
print(f'\nFCI - DMRG: {abs(E_FCI-(E0 + molecule.nuclear_repulsion_energy())):.2e}')

DMRG: -1.137274
HF  : -1.116706
FCI : -1.137274

FCI - HF: 2.06e-02

FCI - DMRG: 4.44e-16


In [ ]:
# Check the number of alpha and beta electrons
print(mps_drv.get_expectation_value(mpo_drv.number_mpo('up')), mps_drv.get_expectation_value(mpo_drv.number_mpo('down'))) 
print(mps_drv.canonical_norm())

(np.float64(1.0000000000000002),
 np.float64(1.0000000000000002),
 np.float64(1.0))

In [ ]:
# Compute the entanglement entropy at the bond/cut between the two sites
mps_drv.bipartite_entang_entropy(0)

np.float64(0.09827671485992703)

In [ ]:
# Compute the 1RDM, which is already diagonal in this simple example
onerdm = mpo_drv.one_rdm(mps_drv)
print('Trace 1RDM = ',np.trace(onerdm))
s,v = np.linalg.eigh(onerdm)[::-1], onerdm

print('1RDM = \n', onerdm)
print(f'1RDM eigvals: {np.linalg.eigvalsh(onerdm)[::-1]}')#, onerdm)

Trace 1RDM =  2.000000000000001
1RDM = 
 [[1.97457654 0.        ]
 [0.         0.02542346]]
1RDM eigvals: [1.97457654 0.02542346]


In [ ]:
# The site occupation numbers can be equivalently obtained as per below
mps_drv.get_expectation_value(mpo_drv.site_occ_mpo(0)), mps_drv.get_expectation_value(mpo_drv.site_occ_mpo(1))

(np.float64(1.9745765353554532), np.float64(0.025423464644547478))